In [1]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from google.colab import drive
drive.mount('/content/drive')
from tensorflow.keras.applications import EfficientNetB0

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
train_real = tf.data.Dataset.list_files("/content/drive/MyDrive/face_detection_train/face_real/*.jpg",shuffle=True)
train_fake = tf.data.Dataset.list_files("/content/drive/MyDrive/face_detection_train/face_fake/*.jpg",shuffle=True)
test_real = tf.data.Dataset.list_files("/content/drive/MyDrive/face_detection_test/face_real/*.jpg",shuffle=True)
test_fake = tf.data.Dataset.list_files("/content/drive/MyDrive/face_detection_test/face_fake/*.jpg",shuffle=True)


In [3]:
type(train_real)

tensorflow.python.data.ops.shuffle_op._ShuffleDataset

In [4]:
def load_image(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    return img
train_real = train_real.map(load_image)
train_fake = train_fake.map(load_image)
test_real = test_real.map(load_image)
test_fake = test_fake.map(load_image)

In [5]:
def add_label(image, label):
  return image , label

train_real = train_real.map(lambda x: add_label(x,0))
train_fake = train_fake.map(lambda x: add_label(x,1))
test_real = test_real.map(lambda x: add_label(x, 0))
test_fake = test_fake.map(lambda x: add_label(x, 1))

In [6]:
train_dataset = train_real.concatenate(train_fake)
test_dataset = test_real.concatenate(test_fake)
train_dataset = train_dataset.shuffle(8000)
test_dataset = test_dataset.shuffle(2000)
train_size = int(0.8 * 7891)   # 6400
val_size = 7891 - train_size   # 1600

validation_dataset = train_dataset.skip(train_size)
train_dataset = train_dataset.take(train_size)

In [ ]:
for image, label in train_dataset.take(5):
    print(image.shape, label)

In [7]:
from tensorflow.keras.applications.efficientnet import preprocess_input
def preprocess(image, label):
    image = preprocess_input(image)
    return image, label
train_dataset = train_dataset.map(preprocess)
validation_dataset = validation_dataset.map(preprocess)
test_dataset = test_dataset.map(preprocess)


In [8]:
BATCH_SIZE = 32
train_dataset = train_dataset.batch(BATCH_SIZE)
validation_dataset = validation_dataset.batch(BATCH_SIZE)
test_dataset = test_dataset.batch(BATCH_SIZE)
AUTOTUNE = tf.data.AUTOTUNE
train_dataset = train_dataset.prefetch(AUTOTUNE)
validation_dataset = validation_dataset.prefetch(AUTOTUNE)
test_dataset = test_dataset.prefetch(AUTOTUNE)

In [9]:
base_model = EfficientNetB0(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [10]:
from tensorflow.keras import layers, Model

inputs = layers.Input(shape=(224, 224, 3))

x = base_model(inputs, training=False)

x = layers.GlobalAveragePooling2D()(x)

x = layers.Dropout(0.2)(x)

outputs = layers.Dense(1, activation="sigmoid")(x)

model = Model(inputs, outputs)

In [11]:
from tensorflow.keras.optimizers import Adam
model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)
history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=10
)

Epoch 1/10
198/198 ━━━━━━━━━━━━━━━━━━━━ 1331s 3s/step - accuracy: 0.5128 - loss: 0.7037 - val_accuracy: 0.5440 - val_loss: 0.6876
Epoch 2/10
198/198 ━━━━━━━━━━━━━━━━━━━━ 562s 3s/step - accuracy: 0.5314 - loss: 0.6949 - val_accuracy: 0.5807 - val_loss: 0.6797
Epoch 3/10
198/198 ━━━━━━━━━━━━━━━━━━━━ 560s 3s/step - accuracy: 0.5531 - loss: 0.6833 - val_accuracy: 0.6099 - val_loss: 0.6684
Epoch 4/10
198/198 ━━━━━━━━━━━━━━━━━━━━ 555s 3s/step - accuracy: 0.5670 - loss: 0.6781 - val_accuracy: 0.6175 - val_loss: 0.6663
Epoch 5/10
198/198 ━━━━━━━━━━━━━━━━━━━━ 586s 3s/step - accuracy: 0.5829 - loss: 0.6732 - val_accuracy: 0.6061 - val_loss: 0.6641
Epoch 6/10
198/198 ━━━━━━━━━━━━━━━━━━━━ 554s 3s/step - accuracy: 0.5881 - loss: 0.6710 - val_accuracy: 0.6244 - val_loss: 0.6584
Epoch 7/10
198/198 ━━━━━━━━━━━━━━━━━━━━ 570s 3s/step - accuracy: 0.5890 - loss: 0.6680 - val_accuracy: 0.6295 - val_loss: 0.6520
Epoch 8/10
198/198 ━━━━━━━━━━━━━━━━━━━━ 514s 2s/step - accuracy: 0.5925 - loss: 0.6671 - val_acc

In [12]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │         1,281 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,053,416 (15.46 MB)

 Trainable params: 1,281 (5.00 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

 Optimizer params: 2,564 (10.02 KB)

In [14]:
model.save("/content/drive/MyDrive/efficientnet_stage1.keras")